# Post-processing — Llama-3.1 and GPT-5.4 outputs

Runs the original post-processing scripts over every Llama-3.1 and GPT-5.4 CSV in `data-generated/`
(`tier1_orig`, `tier1_dsf`, `tier1_medium_only`, `tier1_none_only`, `tier2_orig`, `tier2_dsf`; base and `_retr_evidence` files).
Each script's code (everything above its `main()`) is copied verbatim; its `main()` is replaced by a short run cell that
loops over the files the same way.

| Step | Files | Script | What it does |
|---|---|---|---|
| 1 | GPT-5.4 | `backfill_experiment_dosage_from_answer.py` | Fills empty `gpt4o_dosage` cells from `gpt4o_full` / `gpt4o_answer` |
| 2 | GPT-5.4 | `normalize_gpt54_answer_dosage.py` | `gpt4o_answer` → Yes/No; `gpt4o_dosage` → Low/Medium/High (blank otherwise) |
| 3 | GPT-5.4 | `fix_experiment_dosage_from_full_gpt54Oct16.py` | Re-derives every `gpt4o_dosage` from the reply's own `Dosage:` line or top line (never the explanation); restores `None of the above` |
| 4 | Llama-3.1 | `extract_llama31_dosage_from_full_Oct18.py` | Re-extracts every `gpt4o_dosage` from the target patient's section of multi-patient replies |

**Run top to bottom (Run All).** Several scripts define functions with the same name but different behaviour
(e.g. `_normalize_dosage`), so each step's functions are defined immediately before that step runs.
Steps 1–3 must run in this order: step 2 blanks `None of the above`, and step 3 restores it from the full reply.

Files are modified **in place**, with a timestamped `.bak.<UTC time>` copy next to each changed CSV (as in the original
scripts). Because those `.bak` names only resolve to the second, the notebook also keeps one untouched copy of every
file in `data-generated/raw_before_postprocess/` (made once, never overwritten). Per-row change reports go to `data-generated/postprocess_reports/` (not into the tier folders, so they are
not mistaken for result files). Set `DRY_RUN = True` to preview without writing.

## Setup

In [ ]:
import os
import glob
import pandas as pd

# ==== Configuration ====
DRY_RUN = False          # True = report what would change, write nothing
QUIET = True             # False = print every Llama row (original default; very long for 1,600-row files)
TIER_FOLDERS = ["tier1_orig", "tier1_dsf", "tier1_medium_only", "tier1_none_only", "tier2_orig", "tier2_dsf"]
GPT54_GLOB = "*gpt5_4*.csv"
LLAMA_GLOB = "*llama3_1*.csv"

# ==== Paths ====
# Works whether the notebook sits in the project root or in a subfolder next to data-generated/.
PROJECT_ROOT = os.getcwd() if os.path.isdir(os.path.join(os.getcwd(), "data-generated")) else os.path.dirname(os.getcwd())
OUTPUT_ROOT = os.path.join(PROJECT_ROOT, "data-generated")
REPORT_DIR = os.path.join(OUTPUT_ROOT, "postprocess_reports")
os.makedirs(REPORT_DIR, exist_ok=True)

def _find(pattern):
    paths = []
    for t in TIER_FOLDERS:
        paths += sorted(glob.glob(os.path.join(OUTPUT_ROOT, t, pattern)))
    return [p for p in paths if p.endswith(".csv") and not p.endswith(".dosage_changes.csv")]

GPT54_FILES = _find(GPT54_GLOB)
LLAMA_FILES = _find(LLAMA_GLOB)
rel = lambda p: os.path.relpath(p, PROJECT_ROOT)
print(f"Output root: {OUTPUT_ROOT}")
print(f"\nGPT-5.4 files ({len(GPT54_FILES)}):"); [print("  ", rel(p)) for p in GPT54_FILES]
print(f"\nLlama-3.1 files ({len(LLAMA_FILES)}):"); [print("  ", rel(p)) for p in LLAMA_FILES]


In [ ]:
# ==== Snapshot of the unprocessed files (added; not in the original scripts) ====
# The scripts' .bak copies are named by the second, so when two steps change the same file within one second
# the later .bak overwrites the earlier one and the untouched original is lost. This copies each file once, before
# any step runs, to data-generated/raw_before_postprocess/<tier>/. Existing snapshots are never overwritten,
# so re-running the notebook keeps the true originals.
import shutil
RAW_DIR = os.path.join(OUTPUT_ROOT, "raw_before_postprocess")
n_new = 0
for p in GPT54_FILES + LLAMA_FILES:
    dst = os.path.join(RAW_DIR, os.path.basename(os.path.dirname(p)), os.path.basename(p))
    if not os.path.exists(dst) and not DRY_RUN:
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy2(p, dst)
        n_new += 1
print(f"Raw snapshots: {n_new} new, kept in {rel(RAW_DIR)}")


## Step 1 — GPT-5.4: backfill empty dosage (`backfill_experiment_dosage_from_answer.py`)

In [ ]:
# ==== Verbatim from postprocess/backfill_experiment_dosage_from_answer.py (all code above `def main`) ====
from __future__ import annotations

import argparse
import glob
import os
import re
import shutil
from datetime import datetime, timezone

import numpy as np
import pandas as pd


def _is_empty_dosage(s) -> bool:
    if s is None or (isinstance(s, float) and np.isnan(s)):
        return True
    t = str(s).strip()
    return t == "" or t.lower() == "nan"


def _safe_str(x) -> str:
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return ""
    return str(x)


def _dosage_line_from_blob(blob: str) -> str | None:
    """First 'Dosage: ...' line value (anywhere in blob)."""
    for raw in blob.splitlines():
        line = raw.strip()
        m = re.match(r"^dosage\s*:\s*(.+)$", line, flags=re.IGNORECASE)
        if m:
            val = m.group(1).strip()
            if val:
                return val
    return None


def _infer_dosage_from_compact_answer(text: str) -> str | None:
    """
    Parse dosage embedded in a single-line style answer, e.g.
    'Yes. Low (1 week) supply.' / 'Yes — Low (1 week) supply.' / 'Yes — Medium (2 weeks).'
    Order: longest / most specific first.
    """
    t = _safe_str(text)
    if not t:
        return None
    low = t.lower()

    # Tier 3 / four-way wording
    if re.search(r"none\s+of\s+the\s+above", low):
        return "None of the above"
    if re.search(r"medium\s*\(\s*2\s*weeks?\s*\)", low):
        return "Medium (2 weeks)"
    if re.search(r"high\s*\(\s*4\s*weeks?\s*\)", low):
        return "High (4 weeks)"
    if re.search(r"low\s*\(\s*1\s*week\s*\)", low):
        return "Low (1 week)"

    # Tier 1 style (mg) if present in text
    if re.search(r"low\s*\(\s*0\.5\s*mg\s*\)", low):
        return "Low (0.5 mg)"
    if re.search(r"high\s*\(\s*1\s*mg\s*\)", low):
        return "High (1 mg)"

    return None


def extract_dosage_for_row(answer: str, full_text: str) -> str | None:
    """Return a non-empty dosage string to write, or None if unknown."""
    blob = _safe_str(full_text) + "\n" + _safe_str(answer)
    from_line = _dosage_line_from_blob(blob)
    if from_line and not _is_empty_dosage(from_line):
        return from_line.strip()
    inferred = _infer_dosage_from_compact_answer(_safe_str(answer))
    if inferred:
        return inferred
    # Sometimes only full has compact form without "Dosage:" prefix
    inferred2 = _infer_dosage_from_compact_answer(_safe_str(full_text))
    if inferred2:
        return inferred2
    return None


def _pair_columns(df: pd.DataFrame) -> tuple[str, str, str | None] | None:
    """Return (dosage_col, answer_col, full_col_or_None) for the first *_dosage prefix."""
    dosage_cols = [c for c in df.columns if str(c).lower().endswith("_dosage")]
    if not dosage_cols:
        return None
    dc = dosage_cols[0]
    if len(dosage_cols) > 1:
        # Prefer gpt54_ then gpt4o_
        for pref in ("gpt54_dosage", "gpt4o_dosage"):
            if pref in dosage_cols:
                dc = pref
                break
    prefix = dc[: -len("_dosage")]
    ac = f"{prefix}_answer"
    fc = f"{prefix}_full"
    if ac not in df.columns:
        return None
    full_col = fc if fc in df.columns else None
    return dc, ac, full_col


def backfill_csv(path: str, *, dry_run: bool) -> tuple[int, int]:
    """
    Returns (n_rows_needing_fill, n_rows_filled).
    """
    df = pd.read_csv(path)
    pair = _pair_columns(df)
    if not pair:
        return 0, 0
    dc, ac, fc = pair

    need = df[dc].apply(_is_empty_dosage)
    n_need = int(need.sum())
    if n_need == 0:
        return 0, 0

    filled = 0
    new_vals = df[dc].copy()
    for idx in df.index[need]:
        ans = df.at[idx, ac]
        full = df.at[idx, fc] if fc else ""
        got = extract_dosage_for_row(ans, full)
        if got is not None:
            new_vals.at[idx] = got
            filled += 1

    if filled == 0:
        return n_need, 0

    if not dry_run:
        ts = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
        bak = path + f".bak.{ts}"
        shutil.copy2(path, bak)
        df[dc] = new_vals
        df.to_csv(path, index=False)

    return n_need, filled


In [ ]:
# ---- replaces main(): loop over the GPT-5.4 files ----
total_need = total_filled = 0
for path in GPT54_FILES:
    need, filled = backfill_csv(path, dry_run=DRY_RUN)
    if need:
        print(f"{rel(path)}: rows with empty dosage={need}, backfilled={filled}" + (" [dry-run]" if DRY_RUN else ""))
        total_need += need
        total_filled += filled

print(f"\nTotal: empty dosage rows seen={total_need}, backfilled={total_filled}" + (" [dry-run]" if DRY_RUN else ""))


## Step 2 — GPT-5.4: normalize answer / dosage labels (`normalize_gpt54_answer_dosage.py`)

In [ ]:
# ==== Verbatim from postprocess/normalize_gpt54_answer_dosage.py (all code above `def main`) ====
from __future__ import annotations

import argparse
import glob
import os
import re
import shutil
from datetime import datetime, timezone

import numpy as np
import pandas as pd


def _safe_str(x) -> str:
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return ""
    return str(x)


def _pair_columns(df: pd.DataFrame) -> tuple[str, str, str | None] | None:
    dosage_cols = [c for c in df.columns if str(c).lower().endswith("_dosage")]
    if not dosage_cols:
        return None

    dc = dosage_cols[0]
    if len(dosage_cols) > 1:
        for pref in ("gpt54_dosage", "gpt4o_dosage"):
            if pref in dosage_cols:
                dc = pref
                break

    prefix = dc[: -len("_dosage")]
    ac = f"{prefix}_answer"
    fc = f"{prefix}_full"
    if ac not in df.columns:
        return None
    return dc, ac, fc if fc in df.columns else None


def _normalize_answer(answer: str, full_text: str = "") -> str:
    blob = (_safe_str(answer) + "\n" + _safe_str(full_text)).strip()
    if not blob:
        return ""

    # Prefer explicit "Answer: Yes/No" lines if present.
    for raw in blob.splitlines():
        line = raw.strip()
        m = re.match(r"^\*{0,2}\s*answer\s*:\s*\*{0,2}\s*(yes|no)\b", line, flags=re.IGNORECASE)
        if m:
            return m.group(1).capitalize()

    # Then parse initial compact style ("Yes. ...", "No - ...").
    m0 = re.match(r"^\s*(yes|no)\b", blob, flags=re.IGNORECASE)
    if m0:
        return m0.group(1).capitalize()

    # Fallback: first standalone yes/no occurrence.
    m_any = re.search(r"\b(yes|no)\b", blob, flags=re.IGNORECASE)
    if m_any:
        return m_any.group(1).capitalize()

    return ""


def _normalize_dosage(dosage: str, answer: str = "", full_text: str = "") -> str:
    blob = " ".join([_safe_str(dosage), _safe_str(answer), _safe_str(full_text)]).lower()
    if not blob:
        return ""

    # Respect user's request: only Low / Medium / High labels.
    if re.search(r"\bmedium\b", blob):
        return "Medium"
    if re.search(r"\bhigh\b", blob):
        return "High"
    if re.search(r"\blow\b", blob):
        return "Low"

    # Non-dose choices (e.g. none/n-a) remain blank.
    return ""


def normalize_csv(path: str, *, dry_run: bool) -> dict[str, int]:
    df = pd.read_csv(path)
    pair = _pair_columns(df)
    if not pair:
        return {"rows": 0, "answer_changed": 0, "dosage_changed": 0}

    dc, ac, fc = pair
    full_series = df[fc] if fc else pd.Series([""] * len(df), index=df.index)

    old_answer = df[ac].copy()
    old_dosage = df[dc].copy()

    new_answer = [
        _normalize_answer(a, f)
        for a, f in zip(df[ac], full_series)
    ]
    new_dosage = [
        _normalize_dosage(d, a, f)
        for d, a, f in zip(df[dc], df[ac], full_series)
    ]

    answer_changed = int((old_answer.astype(str).fillna("") != pd.Series(new_answer, index=df.index).astype(str).fillna("")).sum())
    dosage_changed = int((old_dosage.astype(str).fillna("") != pd.Series(new_dosage, index=df.index).astype(str).fillna("")).sum())

    if (answer_changed or dosage_changed) and not dry_run:
        ts = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
        bak = path + f".bak.{ts}"
        shutil.copy2(path, bak)
        df[ac] = new_answer
        df[dc] = new_dosage
        df.to_csv(path, index=False)

    return {
        "rows": int(len(df)),
        "answer_changed": answer_changed,
        "dosage_changed": dosage_changed,
    }


In [ ]:
# ---- replaces main(): loop over the GPT-5.4 files ----
total_files = total_rows = total_a = total_d = 0
for path in GPT54_FILES:
    stats = normalize_csv(path, dry_run=DRY_RUN)
    if stats["rows"] == 0:
        continue
    total_files += 1
    total_rows += stats["rows"]
    total_a += stats["answer_changed"]
    total_d += stats["dosage_changed"]
    tag = " [dry-run]" if DRY_RUN else ""
    print(
        f"{rel(path)}: rows={stats['rows']}, answer_changed={stats['answer_changed']}, "
        f"dosage_changed={stats['dosage_changed']}{tag}"
    )

tag = " [dry-run]" if DRY_RUN else ""
print(
    f"\nProcessed files={total_files}, rows={total_rows}, "
    f"answer_changed={total_a}, dosage_changed={total_d}{tag}"
)


## Step 3 — GPT-5.4: re-derive dosage from the reply (`fix_experiment_dosage_from_full_gpt54Oct16.py`)
One line changed: the `<name>.dosage_changes.csv` report is written to `REPORT_DIR` instead of next to the CSV.

In [ ]:
# ==== Verbatim from postprocess/fix_experiment_dosage_from_full_gpt54Oct16.py (all code above `def main`) ====
from __future__ import annotations

import argparse
import glob
import os
import re
import shutil
from collections import Counter
from datetime import datetime, timezone

import pandas as pd

# Option word at the start of a value: "Low", "**High**", "Medium (2 weeks)", "None of the above".
_WORD_RE = re.compile(r"^[\W_]*(none\s+of\s+the\s+above|low|medium|high)\b", re.I)
# Standalone "Dosage: ..." line, e.g. "**Dosage:** Low (1 week)", "- Dosage - High".
_DOSAGE_LINE_RE = re.compile(
    r"^[\s>*_#`\-•\d.)]*dosage[*_`\s]*[:\-–—][*_`\s]*(?P<val>.+?)\s*$", re.I
)
_YES_NO_RE = re.compile(r"^[\W_]*(yes|no)\b[\s.,:;!\-–—]*", re.I)
_WEEKS_RE = re.compile(r"\(\s*(\d+)\s*weeks?\s*\)", re.I)
_MG_RE = re.compile(r"\(\s*([\d.]+)\s*mg\s*\)", re.I)

_DEFAULT_LONG = {
    "Low": "Low (1 week)",
    "Medium": "Medium (2 weeks)",
    "High": "High (4 weeks)",
    "None of the above": "None of the above",
}
_EXPECTED_WEEKS = {"Low": 1, "Medium": 2, "High": 4}


def _s(x) -> str:
    return "" if x is None or (isinstance(x, float) and pd.isna(x)) else str(x)


def _lines(text: str) -> list[str]:
    text = text.replace("\\r\\n", "\n").replace("\\n", "\n").replace("\r\n", "\n")
    return [l.strip() for l in text.splitlines() if l.strip()]


def _word(value: str) -> str | None:
    m = _WORD_RE.match(value)
    if not m:
        return None
    w = m.group(1).lower()
    return "None of the above" if w.startswith("none") else w.capitalize()


def extract(full, answer) -> tuple[str | None, str, str]:
    """Return (option_word, source, matched_text). option_word is Low/Medium/High/None of the above."""
    texts = [_s(full), _s(answer)]
    for t in texts:  # 1) explicit Dosage: line
        for line in _lines(t):
            m = _DOSAGE_LINE_RE.match(line)
            if m:
                w = _word(m.group("val"))
                if w:
                    return w, "dosage_line", line
    for t in texts:  # 2) compact form in the first two lines only
        for line in _lines(t)[:2]:
            w = _word(_YES_NO_RE.sub("", line, count=1))
            if w:
                return w, "top_line", line
    return None, "unparsed", ""


def _conflict(word: str, matched: str) -> bool:
    """True if the label and the duration disagree, e.g. 'High (1 week)'."""
    m = _WEEKS_RE.search(matched)
    return bool(m and word in _EXPECTED_WEEKS and int(m.group(1)) != _EXPECTED_WEEKS[word])


def _style(series: pd.Series) -> tuple[str, dict[str, str]]:
    """Detect bare vs long labels in the existing column, and the long form used per option."""
    vals = [v.strip() for v in series.dropna().astype(str) if v.strip() and v.strip().lower() != "nan"]
    if not vals:
        return "long", dict(_DEFAULT_LONG)
    long_count = sum("(" in v for v in vals)
    style = "long" if long_count > len(vals) / 2 else "bare"
    long_map = dict(_DEFAULT_LONG)
    per_word: dict[str, Counter] = {}
    for v in vals:
        w = _word(v)
        if w and ("(" in v):
            m = _WEEKS_RE.search(v) or _MG_RE.search(v)
            if m:
                unit = "mg" if "mg" in m.group(0).lower() else ("week" if m.group(1) == "1" else "weeks")
                per_word.setdefault(w, Counter())[f"{w} ({m.group(1)} {unit})"] += 1
    for w, c in per_word.items():
        long_map[w] = c.most_common(1)[0][0]
    return style, long_map


def _find_cols(df: pd.DataFrame) -> tuple[str, str | None, str | None] | None:
    dosage_cols = [c for c in df.columns if str(c).lower().endswith("_dosage")]
    if not dosage_cols:
        return None
    dc = next((p for p in ("gpt54_dosage", "gpt4o_dosage") if p in dosage_cols), dosage_cols[0])
    prefix = dc[: -len("_dosage")]
    fc = f"{prefix}_full" if f"{prefix}_full" in df.columns else None
    ac = f"{prefix}_answer" if f"{prefix}_answer" in df.columns else None
    if not fc and not ac:
        return None
    return dc, fc, ac


def fix_csv(path: str, *, dry_run: bool) -> dict:
    df = pd.read_csv(path)
    cols = _find_cols(df)
    if not cols:
        return {"skipped": "no *_dosage with matching *_full / *_answer"}
    dc, fc, ac = cols
    style, long_map = _style(df[dc])

    new_col = df[dc].astype(object).copy()
    report = []
    unparsed = conflicts = 0
    for idx in df.index:
        word, source, matched = extract(df.at[idx, fc] if fc else "", df.at[idx, ac] if ac else "")
        if word is None:
            unparsed += 1
            continue  # leave the stored value alone; we couldn't read the reply
        is_conflict = _conflict(word, matched)
        conflicts += is_conflict
        label = word if style == "bare" else long_map.get(word, word)
        old = _s(df.at[idx, dc]).strip()
        if old != label:
            new_col.at[idx] = label
            report.append({"row": idx, "old_dosage": old, "new_dosage": label, "source": source,
                           "label_duration_conflict": is_conflict, "matched_text": matched})

    if report and not dry_run:
        ts = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
        shutil.copy2(path, f"{path}.bak.{ts}")
        df[dc] = new_col
        df.to_csv(path, index=False)
        # CHANGED: report written to REPORT_DIR (was: os.path.splitext(path)[0] + ".dosage_changes.csv")
        pd.DataFrame(report).to_csv(os.path.join(REPORT_DIR, os.path.splitext(os.path.basename(path))[0] + ".dosage_changes.csv"), index=False)

    transitions = Counter(f"{r['old_dosage'] or '<empty>'} -> {r['new_dosage']}" for r in report)
    return {"column": dc, "style": style, "rows": len(df), "changed": len(report),
            "unparsed": unparsed, "conflicts": conflicts, "transitions": dict(transitions)}


In [ ]:
# ---- replaces main(): loop over the GPT-5.4 files ----
total = 0
for p in GPT54_FILES:
    r = fix_csv(p, dry_run=DRY_RUN)
    if "skipped" in r:
        print(f"{rel(p)}: SKIP ({r['skipped']})")
        continue
    total += r["changed"]
    tag = " [dry-run]" if DRY_RUN else ""
    print(f"{rel(p)}: {r['column']} ({r['style']} labels) rows={r['rows']} changed={r['changed']} "
          f"unparsed={r['unparsed']} label/duration conflicts={r['conflicts']}{tag}")
    for t, n in sorted(r["transitions"].items(), key=lambda kv: -kv[1]):
        print(f"    {t}: {n}")
print(f"\nTotal rows changed: {total}" + (" [dry-run]" if DRY_RUN else ""))


## Step 4 — Llama-3.1: extract dosage from the patient's own section (`extract_llama31_dosage_from_full_Oct18.py`)
Run with the script's defaults: every row is re-extracted (not `--only-empty`), files are overwritten in place, and a per-row report (`--report-dir`) is written to `REPORT_DIR`.

In [ ]:
# ==== Verbatim from postprocess/extract_llama31_dosage_from_full_Oct18.py (all code above `def main`) ====
from __future__ import annotations

import argparse
import os
import re
import shutil
from datetime import datetime, timezone

import numpy as np
import pandas as pd

DEFAULT_CSVS = [
    os.path.join("experiment_results", "llama31", "results_post_op_llama3_1_tier2_ff_May23.csv"),
    os.path.join("experiment_results", "llama31", "results_post_op_llama3_1_tier3_ff_May19.csv"),
]

ROMAN_TO_DOSAGE = {
    "i": "Low (1 week)",
    "ii": "Medium (2 weeks)",
    "iii": "High (4 weeks)",
    "iv": "None of the above",
}

NAN_DOSAGE_VALUES = frozenset({"nan", "n/a", "na", "not applicable", "none", "no supply"})

# Canonical labels, checked in order (more specific first).
CANONICAL_PATTERNS = [
    (r"none\s+of\s+the\s+above", "None of the above"),
    (r"medium\s*\(\s*2\s*weeks?\s*\)", "Medium (2 weeks)"),
    (r"high\s*\(\s*4\s*weeks?\s*\)", "High (4 weeks)"),
    (r"low\s*\(\s*1\s*weeks?\s*\)", "Low (1 week)"),
    (r"low\s*\(\s*0\.5\s*mg\s*\)", "Low (0.5 mg)"),
    (r"high\s*\(\s*1\s*mg\s*\)", "High (1 mg)"),
]

# A line that opens a new patient section: "**Vignette 3: Willie**", "**Willie**:",
# "Vignette 3: Willie", "### Willie".
SECTION_HEADER_RE = re.compile(
    r"^\s*(?:#{1,6}\s*|\*\*\s*)(?P<h1>[^*\n]{1,80}?)\s*(?:\*\*)?\s*:?\s*$"
    r"|^\s*(?P<h2>vignette\s*\d+\s*[:\-–]\s*[^\n]{1,60})$",
    flags=re.IGNORECASE | re.MULTILINE,
)

# "(iii) High (4 weeks): Yes, ..." -> option marked as chosen.
OPTION_VERDICT_RE = re.compile(
    r"\(\s*(?P<roman>i{1,3}|iv)\s*\)\s*(?P<label>[^:\n]*?)\s*:\s*\**\s*(?P<verdict>[a-z]+)",
    flags=re.IGNORECASE,
)
YES_WORDS = {"yes", "selected", "chosen", "recommended", "correct"}


# ----------------------------------------------------------------------------- helpers
def _project_root() -> str:
    return os.path.abspath(os.path.join(os.path.dirname(__file__), ".."))


def _safe_str(value) -> str:
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return ""
    return str(value)


def _is_empty_dosage(value) -> bool:
    text = _safe_str(value).strip()
    return text == "" or text.lower() == "nan"


def _normalize_dosage(raw: str) -> str:
    """Map extracted text to canonical experiment dosage strings."""
    text = raw.strip().strip("*").strip().rstrip(".")
    if not text:
        return text
    low = text.lower()
    if low in NAN_DOSAGE_VALUES or "not applicable" in low:
        return "nan (not applicable)"
    roman = re.match(r"^\(?\s*(i{1,3}|iv)\s*\)?$", low)
    if roman:
        return ROMAN_TO_DOSAGE[roman.group(1)]
    for pattern, canonical in CANONICAL_PATTERNS:
        if re.search(pattern, low):
            return canonical
    return text


# ----------------------------------------------------------------------------- sections
def _split_sections(full_text: str) -> list[tuple[str, str]]:
    """Split a reply into (header, body) sections. Text before the first header is dropped."""
    headers = list(SECTION_HEADER_RE.finditer(full_text))
    sections = []
    for i, m in enumerate(headers):
        header = (m.group("h1") or m.group("h2") or "").strip()
        end = headers[i + 1].start() if i + 1 < len(headers) else len(full_text)
        sections.append((header, full_text[m.end():end]))
    # Ignore generic headers such as "**Explanation:**" when deciding patient sections.
    return [(h, b) for h, b in sections if not re.fullmatch(r"(explanation|answer|dosage|note)s?", h, re.I)]


def _section_for_name(full_text: str, name: str) -> tuple[str, str]:
    """
    Return (text, how) for the part of full_text that describes this patient.

    how is one of: 'name-section', 'last-section', 'whole-reply'.
    """
    full_text = _safe_str(full_text)
    name = _safe_str(name).strip()
    sections = _split_sections(full_text)

    if name:
        name_re = re.compile(rf"\b{re.escape(name)}\b", re.IGNORECASE)
        matches = [b for h, b in sections if name_re.search(h)]
        if matches:
            return matches[-1], "name-section"

    if len(sections) >= 2:
        # Multi-patient reply without a named header: the target vignette is last.
        return sections[-1][1], "last-section"

    return full_text, "whole-reply"


# ----------------------------------------------------------------------------- parsers
def _dosage_from_dosage_line(blob: str) -> str | None:
    """'Dosage: High (4 weeks)' as its own line or inline before Explanation."""
    m = re.search(r"\bdosage\s*\**\s*:\s*\**\s*([^\n]+?)(?:\s*explanation\s*:|\n|$)", blob, re.IGNORECASE)
    if m:
        value = m.group(1).strip().rstrip(".")
        if value:
            return value
    return None


def _dosage_from_option_list(blob: str) -> str | None:
    """Option list where the chosen option is marked '(iii) High (4 weeks): Yes, ...'."""
    chosen = []
    for m in OPTION_VERDICT_RE.finditer(blob):
        if m.group("verdict").lower() in YES_WORDS:
            label = m.group("label").strip() or ROMAN_TO_DOSAGE[m.group("roman").lower()]
            chosen.append(_normalize_dosage(label))
    chosen = list(dict.fromkeys(chosen))
    return chosen[0] if len(chosen) == 1 else None  # ambiguous if 0 or >1


def _dosage_from_answer_line(blob: str) -> str | None:
    """'Answer: (iii) High (4 weeks)' — requires the parenthesised roman numeral."""
    m = re.search(r"\banswer\s*:\s*\(\s*(i{1,3}|iv)\s*\)\s*([^.\n]*)", blob, re.IGNORECASE)
    if m:
        return m.group(2).strip() or ROMAN_TO_DOSAGE[m.group(1).lower()]
    return None


def _dosage_from_compact_text(blob: str) -> str | None:
    """Short replies like '(ii) Medium (2 weeks)' or 'High (4 weeks)'.

    Skipped for anything that lists several options, since mentioning an
    option is not the same as choosing it.
    """
    blob = blob.strip()
    if not blob or len(blob) > 300:
        return None
    if len(set(re.findall(r"\(\s*(i{1,3}|iv)\s*\)", blob, re.IGNORECASE))) > 1:
        return None
    hits = {canon for pat, canon in CANONICAL_PATTERNS if re.search(pat, blob, re.IGNORECASE)}
    if len(hits) == 1:
        return hits.pop()
    m = re.fullmatch(r"\(\s*(i{1,3}|iv)\s*\)\.?", blob, re.IGNORECASE)
    if m:
        return ROMAN_TO_DOSAGE[m.group(1).lower()]
    return None


PARSERS = (
    ("dosage-line", _dosage_from_dosage_line),
    ("option-marked-yes", _dosage_from_option_list),
    ("answer-line", _dosage_from_answer_line),
    ("compact", _dosage_from_compact_text),
)


def extract_dosage_from_full(name: str, answer: str, full_text: str) -> tuple[str | None, str]:
    """
    Extract a dosage string for one vignette row.

    Returns (dosage or None, method). Only the patient's own section is searched,
    then gpt4o_answer as a last resort.
    """
    section, how = _section_for_name(full_text, name)
    candidates = [(how, section)]
    answer = _safe_str(answer)
    if answer and answer.strip() != section.strip():
        candidates.append(("gpt4o_answer", answer))

    for source, blob in candidates:
        if not blob.strip() or blob.strip().lower() == "nan":
            continue
        for method, parser in PARSERS:
            found = parser(blob)
            if found:
                normalized = _normalize_dosage(found)
                if normalized:
                    return normalized, f"{source}/{method}"
    return None, f"{how}/unresolved"


# ----------------------------------------------------------------------------- CSV
def process_csv(
    path: str, *, dry_run: bool, only_empty: bool, output: str | None, report_path: str | None
) -> pd.DataFrame:
    df = pd.read_csv(path)
    required = {"name", "gpt4o_dosage", "gpt4o_full"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"{path}: missing columns {sorted(missing)}")

    answer_col = "gpt4o_answer" if "gpt4o_answer" in df.columns else None
    rows = df.index[df["gpt4o_dosage"].apply(_is_empty_dosage)] if only_empty else df.index

    records = []
    new_dosage = df["gpt4o_dosage"].astype(object).copy()
    for idx in rows:
        old = _safe_str(df.at[idx, "gpt4o_dosage"])
        extracted, method = extract_dosage_from_full(
            name=df.at[idx, "name"],
            answer=df.at[idx, answer_col] if answer_col else "",
            full_text=df.at[idx, "gpt4o_full"],
        )
        new_dosage.at[idx] = extracted if extracted is not None else np.nan
        records.append(
            {
                "row": idx,
                "vignette_idx": df.at[idx, "vignette_idx"] if "vignette_idx" in df.columns else "",
                "name": df.at[idx, "name"],
                "old_dosage": old,
                "new_dosage": extracted or "",
                "changed": old != (extracted or ""),
                "method": method,
            }
        )

    report = pd.DataFrame(records)
    if report_path:
        os.makedirs(os.path.dirname(report_path) or ".", exist_ok=True)
        report.to_csv(report_path, index=False)

    if not dry_run and len(report):
        df["gpt4o_dosage"] = new_dosage
        out_path = output or path
        if out_path == path:
            stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
            backup = f"{path}.bak.{stamp}"
            shutil.copy2(path, backup)
            print(f"  backup: {backup}")
        df.to_csv(out_path, index=False)
        print(f"  wrote:  {out_path}")
    return report


In [ ]:
# ---- replaces main(): loop over the Llama-3.1 files ----
ONLY_EMPTY = False   # script default: re-extract every row

totals = {"rows": 0, "changed": 0, "unresolved": 0}
for path in LLAMA_FILES:
    base = os.path.splitext(os.path.basename(path))[0]
    report_path = os.path.join(REPORT_DIR, f"{base}_dosage_report.csv")

    print(f"{rel(path)}")
    report = process_csv(
        path, dry_run=DRY_RUN, only_empty=ONLY_EMPTY, output=None, report_path=report_path
    )
    if not QUIET and len(report):
        with pd.option_context("display.width", 200, "display.max_columns", None):
            print(report.drop(columns=["row"]).to_string(index=False))
    n_unres = int((report["new_dosage"] == "").sum()) if len(report) else 0
    n_changed = int(report["changed"].sum()) if len(report) else 0
    print(f"  rows={len(report)}, changed={n_changed}, unresolved={n_unres}" + (" [dry-run]" if DRY_RUN else ""))
    print(f"  report: {rel(report_path)}")
    totals["rows"] += len(report)
    totals["changed"] += n_changed
    totals["unresolved"] += n_unres

print(f"\nTotal: rows={totals['rows']}, changed={totals['changed']}, unresolved={totals['unresolved']}"
      + (" [dry-run]" if DRY_RUN else ""))


## Check — final dosage labels per file

In [ ]:
rows = []
for path in GPT54_FILES + LLAMA_FILES:
    df = pd.read_csv(path)
    counts = df["gpt4o_dosage"].fillna("<empty>").astype(str).value_counts()
    answers = df["gpt4o_answer"].fillna("<empty>").astype(str).value_counts()
    rows.append({"file": rel(path), "n": len(df),
                 "dosage labels": dict(counts.head(8)),
                 "answer labels": dict(answers.head(4))})
pd.set_option("display.max_colwidth", None)
pd.DataFrame(rows)
